# Exploratory Data Analysis

In this notebook we will look into the speedtest data gathered from ookla and draw some useful insights from the dataset. Additional dataset would also be utilized to make some meaningful insights into the appropriately combined wholistic data

## File Exploration

In the following block we see all the files available for this kernel. The relevant libraries are also imported to process the data extracted from the dataset.

In [1]:
%load_ext cudf.pandas

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import regex as re # For String searches
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2020_quarter_02.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2020_quarter_01.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2021_quarter_01.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2021_quarter_03.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2020_quarter_02.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2021_quarter_02.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2020_quarter_03.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2022_quarter_01.csv
/home/dias-benchmarks/notebooks/gk

In [3]:
data = pd.read_csv('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2021_quarter_03.csv')
data.head()

,Name,Number of Records,Devices,Tests,Avg. Avg U Kbps,Avg. Avg D Kbps,Avg Lat Ms,Avg. Pop2005,Rank Upload,Rank Download,Rank Latency
0,Afghanistan,897,"2,312","9,003","3,983","4,823",70,"25,067,407",217,231,33
1,Åland Islands,284,580,"1,341","58,693","82,767",11,0,27,47,215
2,Albania,"7,215","38,905","104,752","17,354","25,803",20,"3,153,731",100,127,159
3,Algeria,"16,056","70,929","413,666","1,508","9,057",43,"32,854,159",234,211,68
4,American Samoa,80,202,900,"14,345","33,078",18,"64,051",119,104,171


In [4]:
data.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               236 non-null    object
 1   Number of Records  236 non-null    object
 2   Devices            236 non-null    object
 3   Tests              236 non-null    object
 4   Avg. Avg U Kbps    236 non-null    object
 5   Avg. Avg D Kbps    236 non-null    object
 6   Avg Lat Ms         236 non-null    int64
 7   Avg. Pop2005       236 non-null    object
 8   Rank Upload        236 non-null    int64
 9   Rank Download      236 non-null    int64
 10  Rank Latency       236 non-null    int64
dtypes: int64(4), object(7)
memory usage: 24.3+ KB


In [5]:
Mobile_df = pd.DataFrame([],columns=data.columns)
Broadband_df = pd.DataFrame([],columns=data.columns)

def col_name_corrections(df,correction_pair):
    if set(df.columns).intersection(set(correction_pair.keys())):
        df.rename(columns=correction_pair,inplace=True)
    return df

for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'):
    for filename in filenames:
        meta_info = filename.split('/')[-1]
        data = pd.read_csv(dirname+'/'+filename,thousands=r',').convert_dtypes()
        data = col_name_corrections(data,{'Number of Record':'Number of Records'})
        data['Year'] = np.int64(re.search('year_(.*)_quarter',meta_info).group(1))
        data['Quarter'] = np.int64(re.search('quarter_(.*).csv',meta_info).group(1))
        if 'mobile' in meta_info:
            Mobile_df = pd.concat([Mobile_df,data])
        else:
            Broadband_df = pd.concat([Broadband_df,data]) 
print(Broadband_df.shape)
print(Mobile_df.shape)
Mobile_df = Mobile_df.astype({'Year':np.int64,'Quarter':np.int64})
Broadband_df = Broadband_df.astype({'Year':np.int64,'Quarter':np.int64})
Mobile_df.sort_values(by=['Year','Quarter'],ascending=[True,True],inplace=True)
Broadband_df.sort_values(by=['Year','Quarter'],ascending=[True,True],inplace=True)

(2597, 13)
(2487, 13)


It can be seen that there are more broadband rows than Mobile rows. This is a point that should be noted each row corresponds to a country's statistics in the particular year and quarter. Missing data indicates lack of speed test data from the country.

In [6]:
Mobile_df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 2487 entries, 0 to 232
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               2487 non-null   object
 1   Number of Records  2487 non-null   int64
 2   Devices            2487 non-null   int64
 3   Tests              2487 non-null   int64
 4   Avg. Avg U Kbps    2487 non-null   int64
 5   Avg. Avg D Kbps    2487 non-null   int64
 6   Avg Lat Ms         2487 non-null   int64
 7   Avg. Pop2005       2487 non-null   int64
 8   Rank Upload        2487 non-null   int64
 9   Rank Download      2487 non-null   int64
 10  Rank Latency       2487 non-null   int64
 11  Year               2487 non-null   int64
 12  Quarter            2487 non-null   int64
dtypes: int64(12), object(1)
memory usage: 286.8+ KB


# -- STEFANOS -- Replicate Data

In [7]:
# factor = 20
factor = 50000
Mobile_df = pd.concat([Mobile_df]*factor, ignore_index=True)
Mobile_df.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 124350000 entries, 0 to 124349999
Data columns (total 13 columns):
 #   Column             Dtype
---  ------             -----
 0   Name               object
 1   Number of Records  int64
 2   Devices            int64
 3   Tests              int64
 4   Avg. Avg U Kbps    int64
 5   Avg. Avg D Kbps    int64
 6   Avg Lat Ms         int64
 7   Avg. Pop2005       int64
 8   Rank Upload        int64
 9   Rank Download      int64
 10  Rank Latency       int64
 11  Year               int64
 12  Quarter            int64
dtypes: int64(12), object(1)
memory usage: 12.7+ GB


In [7]:
# factor = 20
factor = 50000
Broadband_df = pd.concat([Broadband_df]*factor, ignore_index=True)
Broadband_df.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 129850000 entries, 0 to 129849999
Data columns (total 13 columns):
 #   Column             Dtype
---  ------             -----
 0   Name               object
 1   Number of Records  int64
 2   Devices            int64
 3   Tests              int64
 4   Avg. Avg U Kbps    int64
 5   Avg. Avg D Kbps    int64
 6   Avg Lat Ms         int64
 7   Avg. Pop2005       int64
 8   Rank Upload        int64
 9   Rank Download      int64
 10  Rank Latency       int64
 11  Year               int64
 12  Quarter            int64
dtypes: int64(12), object(1)
memory usage: 13.3+ GB


In [9]:
Mobile_df.head()

,Name,Number of Records,Devices,Tests,Avg. Avg U Kbps,Avg. Avg D Kbps,Avg Lat Ms,Avg. Pop2005,Rank Upload,Rank Download,Rank Latency,Year,Quarter
0,Afghanistan,789,1241,3541,3099,6387,70,25067407,220,220,32,2020,1
1,Albania,1626,4513,7228,11414,40238,26,3153731,75,42,197,2020,1
2,Algeria,14174,48699,124160,6150,7724,65,32854159,197,213,35,2020,1
3,American Samoa,23,33,63,8488,22792,62,64051,156,111,38,2020,1
4,Andorra,45,83,110,18160,72764,39,73483,9,6,112,2020,1


In [10]:
unique_countries_broadband = Broadband_df.groupby('Name').count()
unique_countries_broadband.head()

,Number of Records,Devices,Tests,Avg. Avg U Kbps,Avg. Avg D Kbps,Avg Lat Ms,Avg. Pop2005,Rank Upload,Rank Download,Rank Latency,Year,Quarter
Name,,,,,,,,,,,,
Afghanistan,220,220,220,220,220,220,220,220,220,220,220,220
Albania,220,220,220,220,220,220,220,220,220,220,220,220
Algeria,220,220,220,220,220,220,220,220,220,220,220,220
American Samoa,220,220,220,220,220,220,220,220,220,220,220,220
Andorra,220,220,220,220,220,220,220,220,220,220,220,220


In [11]:
unique_countries_mobile = Mobile_df.groupby('Name').count()
unique_countries_mobile.head()

,Number of Records,Devices,Tests,Avg. Avg U Kbps,Avg. Avg D Kbps,Avg Lat Ms,Avg. Pop2005,Rank Upload,Rank Download,Rank Latency,Year,Quarter
Name,,,,,,,,,,,,
Afghanistan,220,220,220,220,220,220,220,220,220,220,220,220
Albania,220,220,220,220,220,220,220,220,220,220,220,220
Algeria,220,220,220,220,220,220,220,220,220,220,220,220
American Samoa,220,220,220,220,220,220,220,220,220,220,220,220
Andorra,220,220,220,220,220,220,220,220,220,220,220,220


## Insights

The following countries do have mobile speedtest data for all the years and quarters, thereby making less than 10 reports (one per quarter).

In [12]:
# Check for missing values
Mobile_df.isna().any()

Name                 False
Number of Records    False
Devices              False
Tests                False
Avg. Avg U Kbps      False
Avg. Avg D Kbps      False
Avg Lat Ms           False
Avg. Pop2005         False
Rank Upload          False
Rank Download        False
Rank Latency         False
Year                 False
Quarter              False
dtype: bool

In [13]:
Broadband_df.isna().any()

Name                 False
Number of Records    False
Devices              False
Tests                False
Avg. Avg U Kbps      False
Avg. Avg D Kbps      False
Avg Lat Ms           False
Avg. Pop2005         False
Rank Upload          False
Rank Download        False
Rank Latency         False
Year                 False
Quarter              False
dtype: bool

In [14]:
Mobile_df.shape

(49740, 13)

In [15]:

unique_countries_mobile[unique_countries_mobile.Year < 10]['Year']

Series([], Name: Year, dtype: int64)

                                                                                                           
                                         Total time elapsed: 0.443 seconds                                 
                                       5 GPU function calls in 0.037 seconds                               
                                       1 CPU function calls in 0.006 seconds                               
                                                                                                           
                                                       Stats                                               
                                                                                                           
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function              ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ NDFrame.__getattr__   │ 1          │ 0.002       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__lt__       │ 1          │ 0.014       │ 0.014       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__getitem__ │ 2          │ 0.011       │ 0.006       │ 0          │ 0.000       │ 0.000       │
│ Series.__repr__       │ 1          │ 0.010       │ 0.010       │ 0          │ 0.000       │ 0.000       │
│ NDFrame._repr_latex_  │ 0          │ 0.000       │ 0.000       │ 1          │ 0.006       │ 0.006       │
└───────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- NDFrame._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=680942;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.

In [16]:
unique_countries_broadband[unique_countries_broadband.Year < 10]['Year']

Series([], Name: Year, dtype: int64)

## Raw Download Speed Visualization

This visualization can be used to show change of values per country. The improvement values cannot be understood by laymen because an improvement of 50 Kbps **national average** (given) means differnt things to different countries based on economy, population, GDP, Infrastructure, etc.

In [ ]:
Mobile_Stats = Mobile_df.groupby('Name').agg(
    Change_Download=('Avg. Avg D Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Upload=('Avg. Avg U Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Latency=('Avg Lat Ms', lambda x: list(x)[-1] - list(x)[0])
)
Broadband_Stats = Broadband_df.groupby('Name').agg(
    Change_Download=('Avg. Avg D Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Upload=('Avg. Avg U Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Latency=('Avg Lat Ms', lambda x: list(x)[-1] - list(x)[0])
)
# fig = px.histogram(Mobile_Stats['Change_Download'],title='Frequency distribution of Mobile Speed change',
#                    labels={'count':'Frequency','value':'$\Delta Speed (Kpbs)$','variable':'property'},
#                    nbins=100)
# fig.show()

# fig = px.histogram(Broadband_Stats['Change_Download'],title='Frequency distribution of Broadband Speed change',
#                    labels={'count':'Frequency','value':'$\Delta Speed (Kpbs)$','variable':'property'},
#                    nbins=100)
# fig.show()
# Total_Stats = pd.concat([Broadband_Stats['Change_Download'],Mobile_Stats['Change_Download']],axis=1)
# Total_Stats.columns=['Mobile','Broadband']

# STEFANOS: Disable plotting
# fig = go.Figure(data=[go.Histogram(x=Broadband_Stats['Change_Download'],opacity=0.65,name='Broadband')])
# fig.add_trace(go.Histogram(x=Mobile_Stats['Change_Download'],opacity=0.65,name='Mobile'))
# fig.update_layout(barmode='overlay',
#                   title='Frequency Distribution of Speed change',
#                   xaxis_title="$\Delta\ Speed\ (Kbps)$", yaxis_title="Number of Countries",
#                   legend_title='Color')
# fig.show()

It can see that most of the countries changed between -5000 Kbps to 5000 kbps. A common graph for all the countries is possible but makes it difficult to understand. Therefore it is better we split the countries into different visualizations, for seperate degrees of change

In [18]:
# STEFANOS: Disable plotting
# px.bar(Mobile_Stats,y='Change_Download',labels={'Name':'Country','Change_Download':'Observed Change'},title='Summary of all changes 2020 Q1 - 2022 Q2 ')

In [20]:
%%time
#cell2
#ImprovedCountries_M = Mobile_Stats[(Mobile_Stats['Change_Download'] < 3000) &
#                                (Mobile_Stats['Change_Download'] >0)]
#px.bar(ImprovedCountries_M,y='Change_Download',labels={'Name':'Country','Change_Download':'Improved Download Speed'},title='Countries that improved download speeds')

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 3000) &
                                (Broadband_Stats['Change_Download'] > 0)]

# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Speed change',
#                   xaxis_title="Country", yaxis_title="Improved Speed (Kbps)",
#                   legend_title='Color')
# fig.show()

                                                                                                           
                                         Total time elapsed: 0.104 seconds                                 
                                       6 GPU function calls in 0.028 seconds                               
                                       0 CPU function calls in 0.000 seconds                               
                                                                                                           
                                                       Stats                                               
                                                                                                           
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function              ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.__getitem__ │ 3          │ 0.010       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__lt__       │ 1          │ 0.003       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__gt__       │ 1          │ 0.011       │ 0.011       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__and__      │ 1          │ 0.005       │ 0.005       │ 0          │ 0.000       │ 0.000       │
└───────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘